# Модуль 04 · Урок 2. GroupBy, agg, transform, filter

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Разделить → посчитать → собрать
`groupby` применяет принцип split–apply–combine: делит строки по ключам, выполняет вычисление внутри каждой группы и возвращает результат. `df.groupby('category')['sales'].sum()` создаёт одну сумму на категорию. Для нескольких ключей используйте список имён `groupby(['city','category'])`; по умолчанию они превращаются в многоуровневый индекс, который можно вернуть в колонки через `reset_index()` или воспользоваться `as_index=False`.

In [ ]:
import pandas as pd
calls = pd.DataFrame({'team':['A','A','B','B'],'minutes':[10,30,5,15],'resolved':[True,True,False,True]})
print(calls.groupby('team',as_index=False).agg(avg_minutes=('minutes','mean'),total=('minutes','sum')))

## 2. agg: несколько агрегаций, понятные имена
`agg(total=('amount','sum'),count=('amount','size'))` — именованная агрегация. Она помогает вернуть DataFrame с читабельными колонками, не требующий переименования многоуровневых заголовков. `sum` пропускает NA; если различие между «в группе только NA» и «реальная сумма 0» важно, рассмотрите `sum(min_count=1)`.

In [ ]:
print(calls.groupby('team').agg(calls=('minutes','size'),p90=('minutes',lambda s:s.quantile(.9))))

## 3. transform: результат той же длины
В отличие от агрегации, `transform()` возвращает значение для **каждой исходной строки**. Это подходит для доли строки в сумме группы, нормализации в группе или заполнения пропусков медианой категории. Часто `transform` полностью заменяет `merge` с агрегированной таблицей.

In [ ]:
products_demo = pd.DataFrame({'group':['x','x','y'], 'value':[2,6,4]})
products_demo['share'] = products_demo.value / products_demo.groupby('group').value.transform('sum')
print(products_demo)

## 4. filter и apply: осторожный выбор
`groupby.filter(lambda g: ...)` оставляет **целые группы**, удовлетворяющие условию (например, где не менее 3 строк). `groupby.apply` запускает пользовательскую функцию, но часто сложнее и медленнее встроенных `agg`/`transform`; в pandas 3.x особенно важно понимать, включаются ли группировочные столбцы в функцию. Для ясности сначала ищите встроенную операцию.

**Проверьте себя:** когда использовать `agg`, а когда `transform`? Почему среднее средних часто ошибочно? Как оставить только группы с минимум двумя записями?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m4_06 · city kpis

Получите DataFrame с колонками `city`, `orders` (size), `units` (sum qty), `avg_price` (mean price); по городам в алфавитном порядке.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_06: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_06 = expected_for("m4_06", df=df)


In [ ]:
# m4_06: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 4 столбца; колонки: `city`, `orders`, `units`, `avg_price`.

```text
     city  orders  units  avg_price
0  Казань       2      1      232.0
1  Москва       2      6      112.0
2   Тверь       2      4      152.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_06", result, _expected_m4_06)

### m4_07 · share in city

Добавьте к каждой строке `city_share` — долю её `price*qty` в суммарной выручке её города. Нулевые городские суммы допускают NaN.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_07: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_07 = expected_for("m4_07", df=df)


In [ ]:
# m4_07: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 11 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `city_share`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment  city_share
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...    0.395639
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка    0.590909
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно    0.604361
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок    1.000000
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                     0.409091
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None    0.000000
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_07", result, _expected_m4_07)

### m4_08 · multi key

Получите DataFrame `city`,`status`,`orders` (число строк), `revenue` (сумма price*qty), сгруппированный по city/status с сортировкой ключей.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_08: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_08 = expected_for("m4_08", df=df)


In [ ]:
# m4_08: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 5 строк, 4 столбца; колонки: `city`, `status`, `orders`, `revenue`.

```text
     city     status  orders  revenue
0  Казань        new       1      0.0
1  Казань       paid       1    317.0
2  Москва       paid       2    642.0
3   Тверь  cancelled       1    171.0
4   Тверь        new       1    247.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_08", result, _expected_m4_08)

### m4_09 · popular cities

Оставьте из исходной таблицы только те города, в которых число строк >= min_count. Метки и порядок строк сохраните.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `min_count` — входной параметр (значение видно в примере); в этом примере: `2`.


**Выполнение:** используйте готовые переменные `df`, `min_count`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_09: используются переменные df, min_count

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

min_count = 2

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_09 = expected_for("m4_09", df=df, min_count=min_count)


In [ ]:
# m4_09: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, min_count
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_09", result, _expected_m4_09)

### m4_10 · rank in city

Добавьте `price_rank`: место цены по убыванию внутри города (метод min, одинаковым ценам одинаковый ранг).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_10: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_10 = expected_for("m4_10", df=df)


In [ ]:
# m4_10: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 11 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `price_rank`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment  price_rank
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...         1.0
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка         1.0
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно         2.0
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок         1.0
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                          2.0
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None         2.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_10", result, _expected_m4_10)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()